In [0]:
from pyspark.sql.functions import col, sum, count, avg, round

orders = spark.table("ecommerce_ops.silver.orders_cleaned")
customers = spark.table("ecommerce_ops.silver.customers_cleaned")
products = spark.table("ecommerce_ops.silver.products_cleaned")
payments = spark.table("ecommerce_ops.silver.payments_cleaned")

# Join all 4 Silver tables into an enriched analytical dataset
enriched_df = (
    orders.alias("o")
    .join(customers.alias("c"), on="customer_id", how="inner")
    .join(products.alias("p"), on="product_id", how="inner")
    .join(payments.alias("pay"), on="order_id", how="inner")
)


# Gold Table 1: Customer Lifetime Value & Spending Summary
customer_spending_df = (
    enriched_df
    .groupBy("c.customer_id", "c.customer_name", "c.customer_segment", "c.country", "c.city")
    .agg(
        count("o.order_id").alias("total_orders"),
        round(sum("o.total_price_usd"), 2).alias("total_spent_usd"),
        round(sum("o.profit_usd"), 2).alias("total_profit_usd"),
        round(avg("o.total_price_usd"), 2).alias("avg_order_value_usd")
    )
)
customer_spending_df.write.format("delta").mode("overwrite").saveAsTable("ecommerce_ops.gold.customer_spending")

# Gold Table 2: Product & Category Sales Performance
product_perf_df = (
    enriched_df
    .groupBy("p.product_id", "p.product_name", "p.category", "p.sub_category", "p.brand")
    .agg(
        count("o.order_id").alias("order_count"),
        sum("o.quantity").alias("units_sold"),
        round(sum("o.total_price_usd"), 2).alias("gross_revenue_usd"),
        round(sum("o.profit_usd"), 2).alias("net_profit_usd")
    )
)
product_perf_df.write.format("delta").mode("overwrite").saveAsTable("ecommerce_ops.gold.product_performance")

# Gold Table 3: Monthly Revenue, Payment Method & Fraud Trends
monthly_summary_df = (
    enriched_df
    .groupBy("o.order_year", "o.order_month", "pay.payment_method")
    .agg(
        count("o.order_id").alias("completed_orders"),
        round(sum("o.total_price_usd"), 2).alias("monthly_revenue_usd"),
        round(sum("o.profit_usd"), 2).alias("monthly_profit_usd"),
        sum(col("pay.is_high_fraud_risk").cast("int")).alias("high_risk_orders")
    )
    .orderBy("o.order_year", "o.order_month")
)
monthly_summary_df.write.format("delta").mode("overwrite").saveAsTable("ecommerce_ops.gold.monthly_sales_summary")

print("Gold Aggregation Complete! All 3 Gold tables are ready for SQL & BI.")


Gold Aggregation Complete! All 3 Gold tables are ready for SQL & BI.
